In [ ]:
import datetime as dt

import xarray as xr
from pyproj import CRS

In [ ]:
from etdataset.dem import get_dem_from_roi
from etdataset.era5 import (
    ERA5Dataset,
)
from etdataset.icos import (
    get_stations_config,
    kelvin_to_celsius,
)
from etdataset.utils import (
    work_area_from_coord_point,
)
from etdataset.validation_temp.metrics import (
    compute_monthly_metrics_per_station,
)
from etdataset.validation_temp.plot import (
    build_station_timeseries,
    load_all_data,
    plot_variable,
)
from etdataset.validation_temp.temperature_rescaling import (
    add_dewpoint_to_ds,
    compute_ah,
    compute_dewpoint_lr,
    compute_lapse_rate_from_2_levels,
    compute_vapor_pressure,
    create_era5_sub_dataset,
    filter_dataset_by_hours,
    filter_dataset_by_location,
    filter_dataset_by_pressure_levels,
    generate_dates,
    generate_hours,
    get_lapse_rate_monthly,
    get_saturation_vapor_pressure,
    get_ta_td_celsius_at_location,
    get_vapor_pressure_monthly,
    prepare_temperature_inputs,
    read_era5_file,
    save_ta_td_csv,
    temperature_rescaling_constant_lapse_rate,
    temperature_rescaling_variable_lapse_rate,
)

In [ ]:
from etdataset.logging import LoggerManager

logger = LoggerManager.get_logger(__name__)


# Prepare inputs

### Prepare dates

Day

In [ ]:
start_date = dt.date(2023, 3, 3)
end_date = dt.date(2024, 3, 3)
step_date = 31

hours

In [ ]:
hour_start = 8
hour_end = 16
hour_step = 4

In [ ]:
list_hours = generate_hours(hour_start, hour_end, hour_step)

### Stations

In [ ]:
G_CST = 9.80665

In [ ]:
cfg_low = get_stations_config("DE-Geb")
cfg_high = get_stations_config("IT-TrF")

In [ ]:
stations = ["DE-Geb", "IT-TrF"]

In [ ]:
mnt_path = "COP-DEM_GLO-30-DGED_S2_tiles"

## Method 1

In [ ]:
output = "method_1"

In [ ]:
for s in stations:
    cfg = get_stations_config(s)
    for d in generate_dates(start_date, end_date, step_date):
        logger.info(f"Current date : {d}")
        era5_xrds = read_era5_file(
            d,
            ERA5Dataset.ERA5,
            "out",
        )
        era5_filtered = filter_dataset_by_hours(era5_xrds, d, list_hours)

        # FOR ERA5 data only ###################################################
        era5_sub = create_era5_sub_dataset(era5_filtered)
        # get ta and td in celsius at station location
        ta, td = get_ta_td_celsius_at_location(era5_sub, cfg)
        ds_era5_grid = xr.Dataset(
            {
                "ta": ta,
                "tdp": td,
            }
        )
        # save data as csv
        save_ta_td_csv(ds_era5_grid, cfg, output, name_dir="csv_era5_grid")

        # FOR ERA5 RESCALED ####################################################
        # get a roi around the station
        roi_bbox_utm, roi_crs_utm = work_area_from_coord_point(
            cfg.lat, cfg.lon, 10000, 10000, CRS.from_epsg(4326)
        )["utm"]
        # get dem from the roi
        dem = get_dem_from_roi(
            roi_bbox=roi_bbox_utm,
            roi_crs=roi_crs_utm,
            base_dir=mnt_path,
            resolution=60,
        )
        new_dem, era5_dem, updated_data = prepare_temperature_inputs(
            data=dem,
            era5_data=era5_filtered,
            dataset=ERA5Dataset.ERA5,
        )
        lr_monthly = get_lapse_rate_monthly(d)
        coeff = get_vapor_pressure_monthly(d)
        dp_lr_monthly = compute_dewpoint_lr(coeff)
        updated = temperature_rescaling_constant_lapse_rate(
            updated_data,
            new_dem,
            era5_filtered,
            era5_dem,
            lr_monthly,
            dp_lr_monthly,
        )
        ta, td = get_ta_td_celsius_at_location(updated, cfg)
        ds_era5_grid = xr.Dataset(
            {
                "ta": ta,
                "tdp": td,
            }
        )
        save_ta_td_csv(ds_era5_grid, cfg, output, name_dir="csv_era5_rescaled")


In [ ]:
data = load_all_data(
    era5_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/method_1/csv_era5_grid",
    era5_rescaled_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/method_1/csv_era5_rescaled",
    icos_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/icos_data",
    stations="all",
)


In [ ]:
stations_ts = build_station_timeseries(
    data,
    start="2023-03-03",
    end="2024-03-03",
)


In [ ]:
plot_variable(stations_ts, "ta")


In [ ]:
plot_variable(stations_ts, "tdp")

In [ ]:
compute_monthly_metrics_per_station(stations_ts, out_dir="method_1/metrics_csv")

## Method 2

In [ ]:
output = "method_2"

In [ ]:
for s in stations:
    cfg = get_stations_config(s)
    for d in generate_dates(start_date, end_date, step_date):
        logger.info(f"Current date : {d}")
        # Get ERA5 pressure file
        era5_xrds = read_era5_file(
            d,
            ERA5Dataset.ERA5PRESSURE,
            "out",
        )
        era5_filtered = filter_dataset_by_hours(era5_xrds, d, list_hours)

        ####### COMPUTE CURRENT LAPSE RATE ##############################################
        # Filter on pressure levels
        era5_filt_pressure = filter_dataset_by_pressure_levels(
            era5_filtered, ["700", "925"]
        )
        # Filter on location
        era5_filt_location = filter_dataset_by_location(
            era5_filt_pressure, cfg.lat, cfg.lon
        )
        # Compute dewpoint temperature
        era5_add_dewpoint = add_dewpoint_to_ds(era5_filt_location)
        # Get lapse rates
        lr_t = compute_lapse_rate_from_2_levels(era5_add_dewpoint)
        lr_td = compute_lapse_rate_from_2_levels(era5_add_dewpoint, "td")

        ############ RESCALING ############################################################
        # get a roi around the station
        roi_bbox_utm, roi_crs_utm = work_area_from_coord_point(
            cfg.lat, cfg.lon, 10000, 10000, CRS.from_epsg(4326)
        )["utm"]
        # get dem from the roi
        dem = get_dem_from_roi(
            roi_bbox=roi_bbox_utm,
            roi_crs=roi_crs_utm,
            base_dir=mnt_path,
            resolution=60,
        )
        # Comparison
        logger.info(f"ICOS's elevation: {cfg.elev}")
        if cfg.elev > 1500:
            # Get Tera_850
            era5_data = filter_dataset_by_pressure_levels(era5_filtered, "850")
            era5_data = add_dewpoint_to_ds(era5_data)
            dataset = ERA5Dataset.ERA5PRESSURE
        elif cfg.elev < 1500:
            # Get Tera_2m
            era5_xrds = read_era5_file(
                d,
                ERA5Dataset.ERA5,
                "out",
            )
            era5_filtered = filter_dataset_by_hours(era5_xrds, d, list_hours)
            era5_data = era5_filtered
            dataset = ERA5Dataset.ERA5

        # prepare inputs for rescaling
        new_dem, era5_dem, updated_data = prepare_temperature_inputs(
            data=dem,
            era5_data=era5_data,
            dataset=dataset,
        )
        # rescaling
        updated = temperature_rescaling_variable_lapse_rate(
            updated_data=updated_data,
            dataset=dataset,
            dem=new_dem,
            era5_data=era5_data,
            era5_dem=era5_dem,
            lr_ta=lr_t,
            lr_tdp=lr_td,
        )
        # Get temperature and dew point temperature
        ta, td = get_ta_td_celsius_at_location(updated, cfg)
        ds_era5_grid = xr.Dataset(
            {
                "ta": ta,
                "tdp": td,
            }
        )
        save_ta_td_csv(ds_era5_grid, cfg, output, name_dir="csv_era5_rescaled")


In [ ]:
data = load_all_data(
    era5_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/method_1/csv_era5_grid",
    era5_rescaled_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/method_2/csv_era5_rescaled",
    icos_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/icos_data",
    stations="all",
)


In [ ]:
stations_ts = build_station_timeseries(
    data,
    start="2023-03-03",
    end="2024-03-03",
)


In [ ]:
plot_variable(stations_ts, "ta")

In [ ]:
plot_variable(stations_ts, "tdp")


## Method 3

In [ ]:
output = "method_3"

In [ ]:
for s in stations:
    cfg = get_stations_config(s)
    for d in generate_dates(start_date, end_date, step_date):
        logger.info(f"Current date : {d}")
        era5_xrds = read_era5_file(
            d,
            ERA5Dataset.ERA5PRESSURE,
            "out",
        )
        era5_filtered = filter_dataset_by_hours(era5_xrds, d, list_hours)

        # Filter on location
        era5_filt_location = filter_dataset_by_location(
            era5_filtered, cfg.lat, cfg.lon
        )

        ############ RESCALING ############################################################
        # get a roi around the station
        roi_bbox_utm, roi_crs_utm = work_area_from_coord_point(
            cfg.lat, cfg.lon, 10000, 10000, CRS.from_epsg(4326)
        )["utm"]
        # get dem from the roi
        dem = get_dem_from_roi(
            roi_bbox=roi_bbox_utm,
            roi_crs=roi_crs_utm,
            base_dir=mnt_path,
            resolution=60,
        )
        new_dem, era5_dem, updated_data = prepare_temperature_inputs(
            data=dem,
            era5_data=era5_filtered,
            dataset=ERA5Dataset.ERA5,
        )
        # Comparison
        logger.info(f"ICOS's elevation: {cfg.elev}")
        if cfg.elev > 1500:
            # Get Lapse rate 700-850
            ####### Filter on pressure levels
            era5_filt_pressure = filter_dataset_by_pressure_levels(
                era5_filt_location, ["700", "850"]
            )
            ####### Compute dewpoint temperature
            era5_add_dewpoint = add_dewpoint_to_ds(era5_filt_pressure)
            ####### Get lapse rates
            lr_t = compute_lapse_rate_from_2_levels(era5_add_dewpoint)
            lr_td = compute_lapse_rate_from_2_levels(era5_add_dewpoint, "td")

            # Get Tera_850
            era5_data = filter_dataset_by_pressure_levels(era5_filtered, "850")
            era5_data = add_dewpoint_to_ds(era5_data)
            dataset = ERA5Dataset.ERA5PRESSURE
        elif cfg.elev < 1500:
            # Get Lapse rate 850-925
            ####### Filter on pressure levels
            era5_filt_pressure = filter_dataset_by_pressure_levels(
                era5_filt_location, ["925", "850"]
            )
            ####### Compute dewpoint temperature
            era5_add_dewpoint = add_dewpoint_to_ds(era5_filt_pressure)
            ####### Get lapse rates
            lr_t = compute_lapse_rate_from_2_levels(era5_add_dewpoint)
            lr_td = compute_lapse_rate_from_2_levels(era5_add_dewpoint, "td")

            # Get Tera_2m
            era5_xrds = read_era5_file(
                d,
                ERA5Dataset.ERA5,
                "out",
            )
            era5_filtered = filter_dataset_by_hours(era5_xrds, d, list_hours)
            era5_data = era5_filtered
            dataset = ERA5Dataset.ERA5
        # prepare inputs for rescaling
        new_dem, era5_dem, updated_data = prepare_temperature_inputs(
            data=dem,
            era5_data=era5_data,
            dataset=dataset,
        )
        # rescaling
        updated = temperature_rescaling_variable_lapse_rate(
            updated_data=updated_data,
            dataset=dataset,
            dem=new_dem,
            era5_data=era5_data,
            era5_dem=era5_dem,
            lr_ta=lr_t,
            lr_tdp=lr_td,
        )

        ta, td = get_ta_td_celsius_at_location(updated, cfg)
        ds_era5_grid = xr.Dataset(
            {
                "ta": ta,
                "tdp": td,
            }
        )
        save_ta_td_csv(ds_era5_grid, cfg, output, name_dir="csv_era5_rescaled")


In [ ]:
data = load_all_data(
    era5_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/method_1/csv_era5_grid",
    era5_rescaled_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/method_3/csv_era5_rescaled",
    icos_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/icos_data",
    stations="all",
)


In [ ]:
stations_ts = build_station_timeseries(
    data,
    start="2023-03-03",
    end="2024-03-03",
)

In [ ]:
plot_variable(stations_ts, "ta")

In [ ]:
plot_variable(stations_ts, "tdp")

## Method 3 bis

In [ ]:
output = "method_3_bis"

In [ ]:
for s in stations:
    cfg = get_stations_config(s)
    for d in generate_dates(start_date, end_date, step_date):
        logger.info(f"Current date : {d}")
        era5_xrds = read_era5_file(
            d,
            ERA5Dataset.ERA5PRESSURE,
            "out",
        )
        era5_filtered = filter_dataset_by_hours(era5_xrds, d, list_hours)

        # Filter on location
        era5_filt_location = filter_dataset_by_location(
            era5_filtered, cfg.lat, cfg.lon
        )
        # GET LR FOR DEWPOINT TEMPERATURE
        era5_pressure_lr_td = era5_filt_location.copy()
        es = get_saturation_vapor_pressure(
            kelvin_to_celsius(era5_pressure_lr_td["t"])
        )
        e = compute_vapor_pressure(era5_pressure_lr_td["r"], es)

        era5_pressure_lr_td["e"] = (
            era5_pressure_lr_td["t"].dims,
            e,
        )

        era5_pressure_lr_td["e"].attrs.update(
            {"units": "Pa", "long_name": "Vapor pressure"}
        )
        a, _, _, r2 = xr.apply_ufunc(
            compute_ah,
            era5_pressure_lr_td["z"],
            era5_pressure_lr_td["e"],
            input_core_dims=[["pressure_level"], ["pressure_level"]],
            output_core_dims=[[], [], [], []],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[float, float, float, float],
        )
        logger.info(f"Current date : {d} and a : {a} (r2 = {r2})")
        lr_td = compute_dewpoint_lr(a)
        lr_tdp = xr.DataArray(
            lr_td,
            coords={"time": era5_pressure_lr_td.time},
            dims=["time"],
        )
        ############ RESCALING ############################################################
        # get a roi around the station
        roi_bbox_utm, roi_crs_utm = work_area_from_coord_point(
            cfg.lat, cfg.lon, 10000, 10000, CRS.from_epsg(4326)
        )["utm"]
        # get dem from the roi
        dem = get_dem_from_roi(
            roi_bbox=roi_bbox_utm,
            roi_crs=roi_crs_utm,
            base_dir=mnt_path,
            resolution=60,
        )
        new_dem, era5_dem, updated_data = prepare_temperature_inputs(
            data=dem,
            era5_data=era5_filtered,
            dataset=ERA5Dataset.ERA5,
        )
        # Comparison
        logger.info(f"ICOS's elevation: {cfg.elev}")
        if cfg.elev > 1500:
            # Get Lapse rate 700-850
            ####### Filter on pressure levels
            era5_filt_pressure = filter_dataset_by_pressure_levels(
                era5_filt_location, ["700", "850"]
            )
            ####### Compute dewpoint temperature
            era5_add_dewpoint = add_dewpoint_to_ds(era5_filt_pressure)
            ####### Get lapse rates
            lr_t = compute_lapse_rate_from_2_levels(era5_add_dewpoint)

            # Get Tera_850
            era5_data = filter_dataset_by_pressure_levels(era5_filtered, "850")
            era5_data = add_dewpoint_to_ds(era5_data)
            dataset = ERA5Dataset.ERA5PRESSURE
        elif cfg.elev < 1500:
            # Get Lapse rate 850-925
            ####### Filter on pressure levels
            era5_filt_pressure = filter_dataset_by_pressure_levels(
                era5_filt_location, ["925", "850"]
            )
            ####### Compute dewpoint temperature
            era5_add_dewpoint = add_dewpoint_to_ds(era5_filt_pressure)
            ####### Get lapse rates
            lr_t = compute_lapse_rate_from_2_levels(era5_add_dewpoint)

            # Get Tera_2m
            era5_xrds = read_era5_file(
                d,
                ERA5Dataset.ERA5,
                "out",
            )
            era5_filtered = filter_dataset_by_hours(era5_xrds, d, list_hours)
            era5_data = era5_filtered
            dataset = ERA5Dataset.ERA5
        # prepare inputs for rescaling
        new_dem, era5_dem, updated_data = prepare_temperature_inputs(
            data=dem,
            era5_data=era5_data,
            dataset=dataset,
        )
        # rescaling
        updated = temperature_rescaling_variable_lapse_rate(
            updated_data=updated_data,
            dataset=dataset,
            dem=new_dem,
            era5_data=era5_data,
            era5_dem=era5_dem,
            lr_ta=lr_t,
            lr_tdp=lr_tdp,
        )

        ta, td = get_ta_td_celsius_at_location(updated, cfg)
        ds_era5_grid = xr.Dataset(
            {
                "ta": ta,
                "tdp": td,
            }
        )
        save_ta_td_csv(ds_era5_grid, cfg, output, name_dir="csv_era5_rescaled")


In [ ]:
data = load_all_data(
    era5_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/method_1/csv_era5_grid",
    era5_rescaled_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/method_3_bis/csv_era5_rescaled",
    icos_folder="/home/mliateni/Bureau/meriem/et-dataset/notebooks/icos_data",
    stations="all",
)


In [ ]:
stations_ts = build_station_timeseries(
    data,
    start="2023-03-03",
    end="2024-03-03",
)


In [ ]:
plot_variable(stations_ts, "ta")

In [ ]:
plot_variable(stations_ts, "tdp")

## Method 4

In [ ]:
output = "method_4"

In [ ]:
for s in stations:
    cfg = get_stations_config(s)

    for d in generate_dates(start_date, end_date, step_date):
        logger.info(f"Current date : {d}")

        logger.info(f"Station elevation: {cfg.elev}")

        era5_xrds = read_era5_file(
            d,
            ERA5Dataset.ERA5PRESSURE,
            "out",
        )

        era5_filtered = filter_dataset_by_hours(era5_xrds, d, list_hours)

        era5_filt_location = filter_dataset_by_location(
            era5_filtered, cfg.lat, cfg.lon
        )

        heights = era5_filt_location["z"] / G_CST
        levels = era5_filt_location.pressure_level

        mask_above = heights > cfg.elev
        idx_above = mask_above.argmax(dim="pressure_level")
        idx_below = idx_above - 1
        idx_below = xr.where(idx_below < 0, 0, idx_below)

        p_high = levels.isel(pressure_level=idx_above)
        p_low = levels.isel(pressure_level=idx_below)

        lowest_height = heights.isel(pressure_level=0)
        use_surface = cfg.elev < lowest_height  # (time)

        era5_dynamic = era5_filt_location.sel(
            pressure_level=xr.concat([p_low, p_high], dim="pair")
        )

        era5_surface_levels = era5_filt_location.sel(pressure_level=[1000, 975])

        era5_two_levels = xr.where(
            use_surface,
            era5_surface_levels,
            era5_dynamic,
        )

        era5_two_levels = add_dewpoint_to_ds(era5_two_levels)

        lr_t = compute_lapse_rate_from_2_levels(era5_two_levels)

        lr_td = compute_lapse_rate_from_2_levels(era5_two_levels, "td")

        logger.info("Lapse rates computed (vectorized)")

        era5_surface = read_era5_file(
            d,
            ERA5Dataset.ERA5,
            "out",
        )

        era5_surface = filter_dataset_by_hours(era5_surface, d, list_hours)

        era5_surface = filter_dataset_by_location(
            era5_surface, cfg.lat, cfg.lon
        )

        era5_pressure_base = era5_filt_location.sel(pressure_level=p_low)

        era5_data = xr.where(
            use_surface,
            era5_surface,
            era5_pressure_base,
        )

        dataset = ERA5Dataset.ERA5PRESSURE  # rescaling fonction ne change pas

        roi_bbox_utm, roi_crs_utm = work_area_from_coord_point(
            cfg.lat, cfg.lon, 10000, 10000, CRS.from_epsg(4326)
        )["utm"]

        dem = get_dem_from_roi(
            roi_bbox=roi_bbox_utm,
            roi_crs=roi_crs_utm,
            base_dir=mnt_path,
            resolution=60,
        )

        new_dem, era5_dem, updated_data = prepare_temperature_inputs(
            data=dem,
            era5_data=era5_data,
            dataset=dataset,
        )

        updated = temperature_rescaling_variable_lapse_rate(
            updated_data=updated_data,
            dataset=dataset,
            dem=new_dem,
            era5_data=era5_data,
            era5_dem=era5_dem,
            lr_ta=lr_t,
            lr_tdp=lr_td,
        )

        ta, td = get_ta_td_celsius_at_location(updated, cfg)

        ds_era5_grid = xr.Dataset(
            {
                "ta": ta,
                "tdp": td,
            }
        )

        save_ta_td_csv(
            ds_era5_grid,
            cfg,
            output,
            name_dir="csv_era5_rescaled",
        )
